# GameTheory-13d : Optimistic Counterfactual Regret Minimization (OFTRL stable-prédictif)

**Navigation** : [<< 13b-Safe-Subgame-Solving-CSharp](GameTheory-13b-Safe-Subgame-Solving-CSharp.ipynb) | [Index](README.md) | [13e-Revision-Croyances-Adversaire >>](GameTheory-13e-Revision-Croyances-Adversaire-Python.ipynb)

> Source primaire : [carnet Python versionné](GameTheory-13d-Optimistic-CFR-Python.ipynb). Les références proviennent du Python, avec des références bibliographiques explicites. CFR et MCCFR sont calculés en C# ; OpenSpiel ne reçoit que les stratégies moyennes et calcule leur exploitabilité via Python.NET.

> **Notebook compagnon** de `GameTheory-13-ImperfectInfo-CFR-Python.ipynb`. Les modèles de Kuhn et les récurrences du GT-13 servent de socle ; la source les charge, le jumeau C# les réimplémente nativement.

## Counterfactual Regret Minimization — variante optimiste

Ce notebook étudie une variante pédagogique inspirée de la famille **optimiste** du CFR (OFTRL / Optimistic Follow-the-Regularized-Leader), telle que formalisée pour les jeux à information imparfaite par :

> Farina, Kroer, Brown, Sandholm, *Stable-Predictive Optimistic Counterfactual Regret Minimization*, **ICML 2019** (PMLR 97).

### Pourquoi un notebook compagnon (13d) plutôt qu'une section du 13 ?

- Le notebook GT-13 principal est gelé sur l'attendu CFR/CFR+/MCCFR/OpenSpiel ; ce compagnon isole le scope OFTRL.
- Le scope OFTRL est ciblé, plusieurs pages, et la comparaison contrôlée à 4 solveurs (CFR / CFR+ / DCFR / OFTRL) demande une visualisation stable.
- La source réutilise les classes du GT-13 et ajoute `OFTRLSolver` ; le port regroupe leurs récurrences dans son moteur natif. Les limites de cette variante sont explicites au §1.

### Objectifs d'apprentissage

- Distinguer les garanties théoriques de l'OFTRL stable-prédictif publié et cette variante pédagogique, qui ne les établit pas.
- Comparer les six donnes par itération à la rotation d'une seule donne ; ces noms de bras ne désignent pas une alternance de joueurs.
- Apprendre à régler empiriquement $(\kappa, \alpha, \beta)$ selon la profondeur d’historique utilisée par le modèle.
- Évaluer honnêtement : dans les sous-jeux de poker de l'article, **DCFR reste empiriquement meilleur** que leur variante optimiste — l'OFTRL n'est pas une amélioration universelle.



## Préambule : socle réutilisé

Le présent notebook réutilise les classes suivantes du notebook principal
`GameTheory-13-ImperfectInfo-CFR-Python.ipynb` :

- `KuhnPoker` (cellule 5)
- `RegretMatcher` (cellule 8)
- `CFRSolver` (cellule 11)
- `CFRPlusSolver(CFRSolver)` (cellule 20)
- `MCCFRSolver(CFRSolver)` (cellule 23, base pour DCFR dans la littérature)

En C#, `KuhnSolver` regroupe les récurrences CFR, CFR+, DCFR pédagogique et optimiste ; `MonteCarloKuhn` reproduit la récurrence échantillonnée. Les historiques, gains et ensembles d'information viennent du socle Python. Ses démonstrations sont réellement exécutées : regret matching RPS1000, CFR/CFR+5000 et MCCFR50000. Le pont Python.NET ne reçoit que les stratégies moyennes pour le calcul OpenSpiel de meilleure réponse ; il n'exécute pas d'entraînement Python. Les générateurs NumPy et System.Random diffèrent pour MCCFR.


In [1]:
#r "nuget: pythonnet, 3.1.0"
#load "../Config/PythonNetLoader.cs"
using Python.Runtime;

PythonNetLoader.Initialize(typeof(PythonEngine).Assembly, "pyspiel");


Installed Packages pythonnet, 3.1.0

In [2]:
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text.Json;
using Python.Runtime;

// Le moteur de Kuhn suit les mêmes historiques et gains que le socle Python.
public class KuhnSolver
{
    public string Mode;
    public int Iterations;
    public double Kappa, Alpha, Beta;
    // Chaque clé combine la carte privée et l’historique public ; les deux cases sont les actions.
    public Dictionary<string, double[]> Regrets = new();
    public Dictionary<string, double[]> Sums = new();
    public Dictionary<string, double[]> Predictions = new();
    // Les six distributions ordonnées sont équiprobables et excluent les cartes répétées.
    public static int[][] Deals = Enumerable.Range(0, 3).SelectMany(a => Enumerable.Range(0, 3).Where(b => b != a).Select(b => new[] { a, b })).ToArray();
    public KuhnSolver(string mode = "CFR", double kappa = 1, double alpha = .5, double beta = 1)
    {
        Mode = mode;
        Kappa = kappa;
        Alpha = alpha;
        Beta = beta;
    }

    public static double[] Slot(Dictionary<string, double[]> dict, string key)
    {
        // Une information jamais rencontrée part sans regret ni masse de stratégie cumulée.
        if (!dict.ContainsKey(key))
            dict[key] = new double[2];
        return dict[key];
    }

    public double[] Strategy(string key)
    {
        // La prédiction ne corrige les regrets que pour le bras optimiste.
        var r = Slot(Regrets, key);
        var pred = Slot(Predictions, key);
        var a = Math.Max(0, r[0] + (Mode == "OFTRL" ? pred[0] : 0));
        var b = Math.Max(0, r[1] + (Mode == "OFTRL" ? pred[1] : 0));
        // Sans regret positif, une distribution uniforme reste une stratégie valide.
        return a + b > 0 ? new[]
        {
            a / (a + b),
            b / (a + b)
        }

        : new[]
        {
            .5,
            .5
        };
    }

    public double[] Average(string key)
    {
        // L’évaluation porte sur la stratégie moyenne, non sur le dernier regret.
        var s = Slot(Sums, key);
        return s[0] + s[1] > 0 ? new[]
        {
            s[0] / (s[0] + s[1]),
            s[1] / (s[0] + s[1])
        }

        : new[]
        {
            .5,
            .5
        };
    }

    // p représente passe ou abandon selon le contexte ; b représente mise ou suivi.
    public static bool Terminal(string h) => h is "pp" or "pbp" or "pbb" or "bp" or "bb";
    public static double Payoff(string h, int[] cards)
    {
        if (h == "pbp")
            return -1;
        if (h == "bp")
            return 1;
        return (cards[0] > cards[1] ? 1 : -1) * (h == "pp" ? 1 : 2);
    }

    public static string Key(string h, int card) => "JQK"[card] + h;
    public double[] Cfr(string h, int[] cards, double[] reach, int depth = 0)
    {
        if (Terminal(h))
        {
            var p = Payoff(h, cards);
            return new[]
            {
                p,
                -p
            };
        }

        // La parité de l’historique détermine le joueur actif à ce nœud.
        int player = h.Length % 2;
        string key = Key(h, cards[player]);
        var strategy = Strategy(key);
        var utilities = new double[2][];
        var node = new double[2];
        // Les variantes optimiste et DCFR reproduisent la propagation scalaire du source.
        for (int a = 0; a < 2; a++)
        {
            // Chaque branche possède sa propre probabilité de portée.
            var childReach = (double[])reach.Clone();
            if (Mode == "OFTRL" || Mode == "DCFR")
            {
                childReach[0] *= strategy[a];
                childReach[1] *= strategy[a];
            }
            else
                childReach[player] *= strategy[a];
            utilities[a] = Cfr(h + (a == 0 ? "p" : "b"), cards, childReach, depth + 1);
            for (int j = 0; j < 2; j++)
                node[j] += strategy[a] * utilities[a][j];
        }

        var r = Slot(Regrets, key);
        var sum = Slot(Sums, key);
        var pred = Slot(Predictions, key);
        for (int a = 0; a < 2; a++)
        {
            // Le regret compare cette action à la valeur sous la stratégie courante.
            double increment = utilities[a][player] - node[player];
            if (Mode == "DCFR")
            {
                // Le compteur au moins égal à un évite un facteur initial indéfini.
                double t = Math.Max(Iterations, 1);
                double factor = Math.Pow(t, 1.5) / (Math.Pow(t, 1.5) + 1);
                r[a] += reach[1 - player] * Math.Max(increment, 0) * factor;
                sum[a] += reach[0] * strategy[a];
            }
            else
            {
                r[a] += reach[1 - player] * increment;
                // CFR+ tronque les regrets cumulés négatifs après leur mise à jour.
                if (Mode == "CFR+")
                    r[a] = Math.Max(r[a], 0);
                if (Mode == "OFTRL")
                    pred[a] += Kappa / (Math.Max(Iterations, 1) + Beta) / (1 + Alpha * depth) * Math.Max(r[a], 0);
                sum[a] += reach[player] * strategy[a];
            }
        }

        return node;
    }

    public double Train(int count)
    {
        // Une itération complète moyenne les six donnes avant d’avancer le compteur.
        double utility = 0;
        for (int t = 0; t < count; t++)
        {
            utility = 0;
            foreach (var deal in Deals)
                utility += Cfr("", deal, new[] { 1.0, 1.0 })[0] / 6;
            Iterations++;
        }

        return utility;
    }

    // Python.NET ne reçoit que les probabilités moyennes ; OpenSpiel évalue la meilleure réponse.
    public double Exploitability()
    {
        var strategies = new Dictionary<string, double[]>();
        foreach (char card in "JQK")
            foreach (var history in new[]
            {
                "",
                "p",
                "b",
                "pb"
            }

            )
                strategies[card + history] = Average(card + history);
        // La sérialisation transmet une table finie sans partager les objets C# avec Python.
        string payload = JsonSerializer.Serialize(strategies);
        using (Py.GIL())
        using (var scope = Py.CreateScope())
        {
            scope.Set("payload", new PyString(payload));
            scope.Exec(@"import json, pyspiel
from open_spiel.python import policy
from open_spiel.python.algorithms import exploitability
values=json.loads(payload)
game=pyspiel.load_game('kuhn_poker')
tab=policy.TabularPolicy(game)
for key in tab.to_dict():
    ours='JQK'[int(key[0])]+key[1:]
    probs=values[ours]
    arr=tab.policy_for_key(key)
    arr[0], arr[1]=probs
result=float(exploitability.exploitability(game,tab))");
            double result = scope.Get("result").As<double>();
            // Une mesure absente ou non finie ne doit pas être présentée comme un résultat.
            if (!double.IsFinite(result))
                throw new InvalidOperationException("Exploitabilité non finie");
            return result;
        }
    }
}

public class MonteCarloKuhn : KuhnSolver
{
    // La graine rend reproductible la suite de distributions et d’actions échantillonnées.
    private readonly Random random = new Random(20261010);
    public double Sample(string h, int[] cards, int explorer)
    {
        if (Terminal(h))
        {
            var payoff = Payoff(h, cards);
            return explorer == 0 ? payoff : -payoff;
        }

        int player = h.Length % 2;
        string key = Key(h, cards[player]);
        var strategy = Strategy(key);
        // Le joueur exploré évalue les deux actions ; son adversaire en échantillonne une.
        if (player == explorer)
        {
            var utilities = new[]
            {
                Sample(h + "p", cards, explorer),
                Sample(h + "b", cards, explorer)
            };
            double value = strategy[0] * utilities[0] + strategy[1] * utilities[1];
            var regrets = Slot(Regrets, key);
            for (int a = 0; a < 2; a++)
                regrets[a] += utilities[a] - value;
            return value;
        }

        // La probabilité du premier choix suffit pour tirer une action binaire.
        int action = random.NextDouble() < strategy[0] ? 0 : 1;
        var sum = Slot(Sums, key);
        for (int a = 0; a < 2; a++)
            sum[a] += strategy[a];
        return Sample(h + (action == 0 ? "p" : "b"), cards, explorer);
    }

    public double Expected(string h, int[] cards)
    {
        if (Terminal(h))
            return Payoff(h, cards);
        var strategy = Average(Key(h, cards[h.Length % 2]));
        return strategy[0] * Expected(h + "p", cards) + strategy[1] * Expected(h + "b", cards);
    }

    public double TrainSampled(int count)
    {
        double measured = 0;
        for (int t = 0; t < count; t++)
        {
            var cards = Deals[random.Next(Deals.Length)];
            Sample("", cards, 0);
            Sample("", cards, 1);
            Iterations++;
            // La valeur moyenne est mesurée périodiquement, pas à chaque échantillon.
            if (t % 100 == 0)
                measured = Deals.Average(d => Expected("", d));
        }

        return measured;
    }
}

// Le témoin de l'arbre vérifie états terminaux, gains et ensemble d'information.
Console.WriteLine($"Kuhn : pp terminal={KuhnSolver.Terminal("pp")}, p terminal={KuhnSolver.Terminal("p")}");
Console.WriteLine($"bb K/J={KuhnSolver.Payoff("bb", new[] { 2, 0 })}, bp J/K={KuhnSolver.Payoff("bp", new[] { 0, 2 })}, clé Q={KuhnSolver.Key("", 1)}");
// Les demonstrations du socle sont executees, y compris les 50 000 echantillons MCCFR.
var rpsRegret = new double[3];
var rpsSum = new double[3];
for (int t = 0; t < 1000; t++)
{
    var positive = rpsRegret.Select(v => Math.Max(0, v)).ToArray();
    double total = positive.Sum();
    var strategy = positive.Select(v => total > 0 ? v / total : 1.0 / 3).ToArray();
    var utilities = new[]
    {
        0.0,
        1.0,
        -1.0
    };
    double expected = Enumerable.Range(0, 3).Sum(a => strategy[a] * utilities[a]);
    for (int a = 0; a < 3; a++)
    {
        rpsRegret[a] += utilities[a] - expected;
        rpsSum[a] += strategy[a];
    }
}

Console.WriteLine($"RPS 1000 : {string.Join(", ", rpsSum.Select(x => x / rpsSum.Sum()))}");
foreach (var mode in new[]
{
    "CFR",
    "CFR+"
}

)
{
    var s = new KuhnSolver(mode);
    double utility = s.Train(5000);
    Console.WriteLine($"{mode} 5000 : utilité {utility:F4}, exploitabilité {s.Exploitability():F6}");
}

var monteCarlo = new MonteCarloKuhn();
Console.WriteLine($"MCCFR 50000 : utilité {monteCarlo.TrainSampled(50000):F4}");


Kuhn : pp terminal=True, p terminal=False


bb K/J=2, bp J/K=1, clé Q=Q


RPS 1000 : 0.0003333333333333333, 0.9993333333333333, 0.0003333333333333333


CFR 5000 : utilité 0.2450, exploitabilité 0.002332


CFR+ 5000 : utilité -0.0677, exploitabilité 0.002881


MCCFR 50000 : utilité -0.0555


## 1. Régime stable-prédictif — séparer théorie et récurrence pédagogique

### 1.1 CFR et regret local

CFR décompose le calcul en minimisateurs de regret aux ensembles d'information. Dans le socle utilisé ici, le **regret matching** normalise les regrets cumulés positifs ; ce n'est pas une mise à jour FTRL à régularisation entropique. Pour un ensemble I, le regret cumulé s'écrit

$$R_i^T(I)=\max_{a\in A(I)}\sum_{t=1}^T r_i^t(I,a).$$

Sous les hypothèses usuelles de gains bornés et de rappel parfait, l'analyse CFR classique contrôle le regret cumulé en ordre $\sqrt{T}$, puis l'erreur de la stratégie moyenne en ordre $T^{-1/2}$. Ces deux grandeurs ne doivent pas être confondues.

### 1.2 Le pas optimiste

Une écriture générique d'OFTRL sur le simplexe est

$$x_{t+1}\in\arg\min_{x\in\Delta}\left\{R(x)+\eta\left\langle\sum_{s=1}^{t}\ell_s+m_{t+1},x\right\rangle\right\},$$

où R est un régulariseur, les vecteurs $\ell_s$ sont les pertes passées et $m_{t+1}$ prédit la perte suivante. Le régulariseur et la position de la prédiction sont essentiels ; une simple somme de prédictions passées n'est pas cette formule.

Farina et al. (2019), dans leur article publié à ICML/PMLR97, annoncent un **taux de convergence vers Nash** $\mathcal{O}(T^{-3/4})$ pour leur construction stable-prédictive, avec stabilité relative à la position dans l'arbre. Ce n'est pas une borne décroissante sur le regret cumulé R_i^T, et ce taux ne découle pas des trois paramètres pédagogiques κ, α, β ci-dessous.

La récurrence du carnet ajoute à ses regrets cumulés une mémoire pondérée de regrets positifs, avec amplitude κ/(t+β) et amortissement 1/(1+α·depth). Elle normalise ensuite la partie positive de regret + mémoire. C'est une variante pédagogique mesurée ; elle ne résout pas l'argmin entropique ci-dessus et ne revendique pas les garanties du solveur publié.

Les variantes optimiste et DCFR du source propagent le vecteur complet de probabilités d’atteinte en le multipliant par la probabilité de l’action choisie. Le CFR de base ne modifie que la composante du joueur courant. Le port conserve cette différence historique ; elle est une raison supplémentaire de ne pas attribuer aux variantes pédagogiques les garanties des algorithmes de référence.


### 1.3 Le bémol empirique

L'introduction de l'article rapporte de meilleures performances que CFR avec regret matching sur ses sous-jeux de poker, mais ne dépasse pas DCFR. Le banc ci-dessous calcule l'**exploitabilité réelle** de quatre récurrences sur Kuhn poker ; son classement porte sur ces implémentations et ce budget, sans prétendre reproduire les expériences de l'article. Distinguer une garantie publiée, une réputation empirique et une mesure locale est l'objet de la section2.


In [3]:
// Démonstration sur la donne [J,Q], puis contrôle à compteurs alignés.
KuhnSolver[] NewSolvers() => new[]
{
    new KuhnSolver("CFR"),
    new KuhnSolver("CFR+"),
    new KuhnSolver("DCFR"),
    new KuhnSolver("OFTRL")
};
void Step(KuhnSolver[] solvers, int t, IEnumerable<int[]> deals)
{
    // Tous les solveurs voient la même donne et le même compteur pour une comparaison alignée.
    foreach (var deal in deals)
        foreach (var solver in solvers)
        {
            solver.Iterations = t;
            solver.Cfr("", deal, new[] { 1.0, 1.0 });
        }
}

var demo = NewSolvers();
for (int t = 1; t <= 500; t++)
{
    Step(demo, t, new[] { new[] { 0, 1 } });
    if (new[]
    {
        1,
        10,
        50,
        100,
        200,
        300,
        400,
        500
    }.Contains(t))
        Console.WriteLine($"Démo {t} : " + string.Join(" ; ", demo.Select(s => s.Mode + " Qp=" + string.Join(",", s.Average("Qp").Select(x => x.ToString("F4"))))));
}


Démo 1 : CFR Qp=0.5000,0.5000 ; CFR+ Qp=0.5000,0.5000 ; DCFR Qp=0.5000,0.5000 ; OFTRL Qp=0.5000,0.5000


Démo 10 : CFR Qp=0.0500,0.9500 ; CFR+ Qp=0.0500,0.9500 ; DCFR Qp=0.0338,0.9662 ; OFTRL Qp=0.0314,0.9686


Démo 50 : CFR Qp=0.0100,0.9900 ; CFR+ Qp=0.0100,0.9900 ; DCFR Qp=0.0056,0.9944 ; OFTRL Qp=0.0052,0.9948


Démo 100 : CFR Qp=0.0050,0.9950 ; CFR+ Qp=0.0050,0.9950 ; DCFR Qp=0.0027,0.9973 ; OFTRL Qp=0.0026,0.9974


Démo 200 : CFR Qp=0.0025,0.9975 ; CFR+ Qp=0.0025,0.9975 ; DCFR Qp=0.0013,0.9987 ; OFTRL Qp=0.0013,0.9987


Démo 300 : CFR Qp=0.0017,0.9983 ; CFR+ Qp=0.0017,0.9983 ; DCFR Qp=0.0009,0.9991 ; OFTRL Qp=0.0008,0.9992


Démo 400 : CFR Qp=0.0013,0.9988 ; CFR+ Qp=0.0013,0.9988 ; DCFR Qp=0.0006,0.9994 ; OFTRL Qp=0.0006,0.9994


Démo 500 : CFR Qp=0.0010,0.9990 ; CFR+ Qp=0.0010,0.9990 ; DCFR Qp=0.0005,0.9995 ; OFTRL Qp=0.0005,0.9995


### Exercice 1 — Régler $(\kappa, \alpha, \beta)$ par profondeur

**Énoncé** : Sur le modèle de Kuhn Poker du carnet, la variable `depth` compte les actions déjà jouées : 0,1 ou2 aux ensembles d’information, et jusqu’à3 aux états terminaux. Ce compteur d’historique ne définit pas la profondeur théorique d’un treeplex. Ajuster $(\kappa, \alpha, \beta)$ pour maximiser la convergence de l'OFTRL en 100 itérations sur le mode `simultaneous`. Mesurer la vitesse avec l'exploitabilité réelle (`Exploitability()`, meilleure réponse), et non avec une distance à l'uniforme qui classe mal les stratégies.

**Indices** :
- `depth` est passé récursivement à `Cfr()` ; il compte la longueur de l’historique d’actions
- κ=0 annule la mémoire prédictive ; α et β la modulent différemment et ne doivent pas être assimilés à son amplitude
- Mesurer les deux configurations au budget demandé ; aucune monotonie de performance en κ, α ou β n’est supposée
- Les configurations pédagogiques de cet exercice incluent $\kappa \approx 1.5$, $\alpha \approx 0.3$, $\beta \approx 0.5$

Pour rendre la comparaison reproductible, utiliser les triplets (1, 0.5, 1) et (1.5, 0.3, 0.5) et renvoyer le meilleur avec sa mesure.


In [4]:
// Exercice ouvert : choisir le meilleur triplet sur la grille annoncée.
Func<((double k, double a, double b) parameters, double score)?> exercice1 = () => null;
var reponse1 = exercice1();
if (reponse1 == null)
    Console.WriteLine("Exercice 1 à compléter.");
else
{
    var candidats = new[]
    {
        (1.0, .5, 1.0),
        (1.5, .3, .5)
    };
    var scores = new List<double>();
    foreach (var candidate in candidats)
    {
        var solver = new KuhnSolver("OFTRL", candidate.Item1, candidate.Item2, candidate.Item3);
        for (int t = 1; t <= 100; t++)
        {
            solver.Iterations = t;
            foreach (var deal in KuhnSolver.Deals)
                solver.Cfr("", deal, new[] { 1.0, 1.0 });
        }

        scores.Add(solver.Exploitability());
    }

    int index = Array.IndexOf(candidats, reponse1.Value.parameters);
    if (index < 0 || !double.IsFinite(reponse1.Value.score) || Math.Abs(reponse1.Value.score - scores[index]) > 1e-9 || reponse1.Value.score > scores.Min() + 1e-9)
        throw new Exception("Triplet ou exploitabilité incorrects");
    Console.WriteLine("Exercice 1 : sondes réussies.");
}


Exercice 1 à compléter.


### 1.4 DCFR publié et variante pédagogique mesurée

DCFR de référence applique une pondération aux regrets cumulés passés et à la stratégie moyenne. La variante versionnée ici est différente ; son nom historique ne suffit pas à lui transférer ces garanties. Le banc exécute précisément :

- le facteur $t^{1.5}/(t^{1.5}+1)$ sur **l'incrément positif** de regret ; ce facteur tend vers1, sans décroître le cumul déjà acquis ;
- une contribution à la stratégie moyenne sans pondération temporelle supplémentaire dans la configuration β=0 du banc Python, reproduite en C# ;
- un écrêtage de chaque incrément par max(r,0), systématique. Cela diffère de CFR+, qui écrête la somme mise à jour. Dans la source, le paramètre gamma est stocké mais inactif ; le port reproduit cette configuration fixe.

La cellule suivante lance les mêmes contrôles à compteurs remis à zéro sur la donne J/Q. Les quatre solveurs ont reçu le même nombre de mises à jour à chaque jalon ; la comparaison porte sur Qp, ensemble d'information du second joueur tenant Q après un check. Ce diagnostic local ne mesure pas, à lui seul, l'exploitabilité de toute la stratégie.

Une implémentation qui pondère l'ancien cumul au lieu du seul incrément peut produire d'autres résultats. Le benchmark de la section2 compare la variante effectivement écrite, sans conclure que DCFR de référence serait mauvais en général.


In [5]:
// Solveurs neufs : les jalons 10/100/500 ont les mêmes compteurs sur la donne J/Q.
var sanity = NewSolvers();
for (int t = 1; t <= 500; t++)
{
    Step(sanity, t, new[] { new[] { 0, 1 } });
    if (new[]
    {
        10,
        100,
        500
    }.Contains(t))
        Console.WriteLine($"Sanity {t} : " + string.Join(" ; ", sanity.Select(s => s.Mode + " Qp=" + string.Join(",", s.Average("Qp").Select(x => x.ToString("F4"))))));
}


Sanity 10 : CFR Qp=0.0500,0.9500 ; CFR+ Qp=0.0500,0.9500 ; DCFR Qp=0.0338,0.9662 ; OFTRL Qp=0.0314,0.9686


Sanity 100 : CFR Qp=0.0050,0.9950 ; CFR+ Qp=0.0050,0.9950 ; DCFR Qp=0.0027,0.9973 ; OFTRL Qp=0.0026,0.9974


Sanity 500 : CFR Qp=0.0010,0.9990 ; CFR+ Qp=0.0010,0.9990 ; DCFR Qp=0.0005,0.9995 ; OFTRL Qp=0.0005,0.9995


### 1.5 L'instrument de mesure : la meilleure réponse, pas une distance à l'uniforme

Une comparaison honnête exige de neutraliser tout sauf l'opérateur de mise à jour : **même budget d'itérations**, **mêmes donnes**, **mêmes adversaires**. La cellule suivante instancie les quatre variantes — CFR, CFR+, DCFR, OFTRL — sur Kuhn poker et rapporte deux mesures : le **temps de calcul** (`Stopwatch`) et l'**exploitabilité réelle**, c'est-à-dire ce qu'un meilleur-respondant optimal peut encore gagner contre la stratégie moyenne (`Exploitability()` du socle, qui délègue le calcul de meilleure réponse à OpenSpiel). Zéro = équilibre de Nash.

Deux précautions que ce carnet a dû corriger :

- **la distance à la stratégie uniforme n'est pas une exploitabilité.** Elle mesure l'éloignement de $[0{,}5, 0{,}5]$, pas l'éloignement de Nash — et sur l'info-set `Q` ces deux distances se contredisent : la stratégie de Nash y est $[1, 0]$ (ne jamais miser), donc une stratégie à $[0{,}9995, 0{,}0005]$ est **quasi parfaite** tout en étant très loin de l'uniforme. Classer les solveurs sur cette distance retourne le verdict ;
- **les quatre solveurs sont déterministes.** Aucun d'eux ne tire de nombre au hasard : `np.random.seed` ne les atteint pas. Un axe « 4 graines » produirait huit lignes identiques à l'octet et ne mesurerait aucune variation — la cellule le vérifie et le dit, au lieu de le suggérer.

Les donnes sont énumérées **en entier** (six donnes, pas trois) : avec trois donnes seulement, les info-sets où le joueur 2 détient un roi ne sont jamais visités, gardent leur stratégie uniforme initiale, et l'exploitabilité plafonne quel que soit le budget. La cellule suivante mesure cet effet au lieu de le supposer.

La différence de travail entre les deux bras est volontaire : six donnes contre une par itération. Le test de déterminisme concerne ces quatre récurrences exhaustives ; il ne concerne ni MCCFR ni les tirages du mini-jeu de l’exercice3.


In [6]:
// Chaque bras conserve le budget de 1000 itérations ; la rotation ne traite qu'une donne.
foreach (string mode in new[]
{
    "simultaneous",
    "alternating"
}

)
{
    var solvers = NewSolvers();
    var sw = System.Diagnostics.Stopwatch.StartNew();
    for (int t = 1; t <= 1000; t++)
        Step(solvers, t, mode == "simultaneous" ? KuhnSolver.Deals : new[] { KuhnSolver.Deals[(t - 1) % 6] });
    sw.Stop();
    foreach (var solver in solvers)
        Console.WriteLine($"{mode} {solver.Mode} : {solver.Exploitability():F6}, Q=[{string.Join(",", solver.Average("Q").Select(x => x.ToString("F4")))}], temps total={sw.Elapsed.TotalSeconds:F3}s");
}

var controls = new List<string>();
foreach (int seed in new[]
{
    0,
    42
}

)
{
    var solvers = NewSolvers();
    for (int t = 1; t <= 300; t++)
        Step(solvers, t, KuhnSolver.Deals);
    controls.Add(string.Join(";", solvers.Select(s => string.Join(",", s.Average("Q").Select(x => x.ToString("F6"))))));
}

Console.WriteLine($"Déterminisme : {controls[0] == controls[1]}");
foreach (var deals in new[]
{
    new[]
    {
        new[]
        {
            0,
            1
        },
        new[]
        {
            1,
            0
        },
        new[]
        {
            2,
            1
        }
    },
    KuhnSolver.Deals
}

)
{
    var solvers = NewSolvers();
    for (int t = 1; t <= 1000; t++)
        Step(solvers, t, deals);
    Console.WriteLine($"{deals.Length} donnes : " + string.Join(" ; ", solvers.Select(s => s.Mode + "=" + s.Exploitability().ToString("F6"))));
}


simultaneous CFR : 0.006505, Q=[0.9988,0.0012], temps total=0.192s


simultaneous CFR+ : 0.006784, Q=[0.9889,0.0111], temps total=0.192s


simultaneous DCFR : 0.071657, Q=[0.6208,0.3792], temps total=0.192s


simultaneous OFTRL : 0.018525, Q=[0.9986,0.0014], temps total=0.192s


alternating CFR : 0.018506, Q=[0.9929,0.0071], temps total=0.015s


alternating CFR+ : 0.019493, Q=[0.9655,0.0345], temps total=0.015s


alternating DCFR : 0.073547, Q=[0.6184,0.3816], temps total=0.015s


alternating OFTRL : 0.022055, Q=[0.9924,0.0076], temps total=0.015s


Déterminisme : True


3 donnes : CFR=0.193655 ; CFR+=0.193131 ; DCFR=0.312279 ; OFTRL=0.211055


6 donnes : CFR=0.006505 ; CFR+=0.006784 ; DCFR=0.071657 ; OFTRL=0.018525


## 2. Verdict honnête

Le benchmark précédent **retourne** la lecture attendue au §1.3 : à budget d'itérations égal, sur Kuhn poker, le **DCFR est dernier** — dans les deux bras, et de loin.

| Solveur | Exploitabilité, `simultaneous` | `alternating` | Stratégie apprise sur `Q` (Nash : `[1, 0]`) |
|---|---|---|---|
| CFR | **0,006505** | 0,018506 | `[0,9988, 0,0012]` |
| CFR+ | 0,006784 | 0,019493 | `[0,9889, 0,0111]` |
| OFTRL | 0,018525 | 0,022055 | `[0,9986, 0,0014]` |
| DCFR | 0,071657 | 0,073547 | `[0,6208, 0,3792]` |

Ce que ces quatre lignes disent, et ce qu'elles ne disent pas :

- **DCFR est onze fois plus exploitable que CFR** (0,0717 contre 0,0065) et sa stratégie sur `Q` mise **38 % du temps avec une dame**, là où l'équilibre ne mise jamais. Ce n'est pas une nuance de classement, c'est un solveur qui n'a pas convergé.
- **CFR+ ne devance pas CFR** ici (0,0068 contre 0,0065) : l'écart déterministe mesuré vaut environ 0,000279. Ce classement à budget fixé ne s'extrapole pas à d'autres horizons ou jeux.
- **OFTRL non réglé est intermédiaire** (0,0185) : cette configuration n'est pas la plus performante à ce budget, sans isoler causalement l'effet de chacun des paramètres — mais il reste **meilleur que le DCFR** sur ce banc.
- **Le bras `alternating` ne fait pas ralentir les solveurs : il fait moins de travail.** Il parcourt une donne par itération au lieu de six, ce qui réduit le travail mesuré, et produit une exploitabilité plus haute à nombre d'itérations égal. L'ancienne prose y lisait « l'avantage optimiste effacé » : il n'y a rien à effacer, seulement six fois moins de mises à jour.

**Ce que ce verdict ne prétend pas.** Il mesure **une implémentation** : celle du §1.4, dont les facteurs portent sur l'incrément de regret et ne décroissent pas la somme accumulée. Une implémentation de référence — où l'ancien est effectivement oublié — n'est pas mesurée ici, et rien dans ce carnet ne dit que le DCFR serait mauvais en général. C'est la distinction que le carnet enseigne désormais : **la réputation d'un algorithme ne se transporte pas d'un code à l'autre**, et « l'article l'affirme » n'est pas « je le mesure ici ».

## 3. Exercices

Les trois activités sont distribuées au fil du carnet :

1. régler les paramètres en fonction de la profondeur d’historique du modèle : l’énoncé, les indices et la cellule ouverte de l’exercice 1 se trouvent dans la section 1 ;
2. comparer les modes simultanés et alternés sur DCFR, avec le budget et les donnes fixés : exercice 2 ci-dessous ;
3. construire la prédiction contrefactuelle sur le petit jeu synthétique, sans modifier son historique : exercice 3 ci-dessous.

Les cellules restent à compléter ; ce récapitulatif ne donne aucune solution.


### Exercice 2 — Modes simultanés vs alternés sur DCFR

**Énoncé** : Comparer le comportement de DCFR dans les deux bras du benchmark — `simultaneous` (chaque itération parcourt les six donnes) et `alternating` (chaque itération ne parcourt qu'une donne, en rotation, donc six fois moins de mises à jour). Tracer la courbe d'exploitabilité réelle (`Exploitability()`) en fonction des itérations, pour 1000 itérations. Les solveurs étant déterministes, il n'y a pas de graine à faire varier : ce qui varie est le nombre de mises à jour par itération.

**Indices** :
- Mesurer les deux plans sur les mêmes 20 jalons (tous les 50 tours).
- Comparer les résultats sans attribuer de variation aux graines inactives.

La fonction rend les vingt couples (itération, exploitabilité) de chaque bras ; tracer ensuite ces deux séries fait partie de l’exercice. Le garde vérifie les jalons et les valeurs, sans fournir le tracé étudiant.


In [7]:
// Exercice ouvert : deux courbes de vingt jalons, mesurées avec OpenSpiel.
Func<Dictionary<string, List<(int tour, double exploitabilite)>>> exercice2 = () => null;
var reponse2 = exercice2();
if (reponse2 == null)
    Console.WriteLine("Exercice 2 à compléter.");
else
{
    if (reponse2.Count != 2 || !reponse2.ContainsKey("simultaneous") || !reponse2.ContainsKey("alternating"))
        throw new Exception("Plans incomplets");
    foreach (var mode in new[]
    {
        "simultaneous",
        "alternating"
    }

    )
    {
        // Recalculer les jalons empêche de valider une courbe simplement plausible.
        var solver = new KuhnSolver("DCFR");
        var points = reponse2[mode];
        if (points == null || points.Count != 20)
            throw new Exception("Jalons incomplets");
        for (int t = 1; t <= 1000; t++)
        {
            solver.Iterations = t;
            var deals = mode == "simultaneous" ? KuhnSolver.Deals : new[]
            {
                KuhnSolver.Deals[(t - 1) % 6]
            };
            foreach (var deal in deals)
                solver.Cfr("", deal, new[] { 1.0, 1.0 });
            if (t % 50 == 0)
            {
                var point = points[t / 50 - 1];
                if (point.tour != t || !double.IsFinite(point.exploitabilite) || Math.Abs(point.exploitabilite - solver.Exploitability()) > 1e-9)
                    throw new Exception("Courbe incorrecte");
            }
        }
    }

    Console.WriteLine("Exercice 2 : sondes réussies.");
}


Exercice 2 à compléter.


### Exercice 3 — Prédiction contrefactuelle sur petit jeu synthétique

**Énoncé** : Créer un jeu à 2 joueurs à information imparfaite avec 3 actions et 4 informations sets. Implémenter `OFCFPredictor` (prédicteur pédagogique, sans identité revendiquée avec un algorithme publié) qui produit la **prédiction** $m_{t+1}$ à partir des 5 dernières observations de regret. Comparer la convergence à CFR vanilla sur ce jeu.

**Indices** :
- Définir la classe `MiniGame3x4(NUM_ACTIONS=3, NUM_INFO_SETS=4)` avec `Payoff(history, card)`
- $m_{t+1}$ = moyenne des 5 derniers regrets positifs
- Éprouver si cette prédiction améliore le regret cumulé mesuré $R^T$, sans présumer une accélération

La prédiction consulte les cinq dernières observations déjà présentes dans le dictionnaire de listes passé au prédicteur ; elle ne modifie pas cet historique.

Le squelette donne quatre ensembles et trois actions, avec une application historique/carte déterministe partagée par les deux langues. Les gains aléatoires de graine0 restent propres à leur générateur. Les sondes vérifient le prédicteur sur les quatre ensembles, des historiques courts/vides et une fenêtre de cinq regrets ; elles ne prouvent pas la convergence d’un solveur de ce mini-jeu. La comparaison CFR demandée reste un travail ouvert de modélisation et d’entraînement, au-delà de ce contrat local.


In [8]:
// Mini-jeu synthétique : 3 actions, 4 ensembles d'information et gains fixés par graine.
public class MiniGame3x4
{
    public const int NumActions = 3;
    public const int NumInfoSets = 4;
    private readonly double[, ] payoffs = new double[4, 3];
    public MiniGame3x4()
    {
        var random = new Random(0);
        for (int i = 0; i < 4; i++)
            for (int a = 0; a < 3; a++)
                payoffs[i, a] = 2 * random.NextDouble() - 1;
    }

    public int InfoSet(string history, int card)
    {
        int hash = card;
        foreach (char action in history)
            hash = (hash * 31 + action) & 0x7fffffff;
        return hash % NumInfoSets;
    }

    public double[] Payoff(string history, int card) => Payoff(InfoSet(history, card));
    public double[] Payoff(int info) => Enumerable.Range(0, 3).Select(a => payoffs[info, a]).ToArray();
    public bool Terminal(string history) => history.Length >= 2;
    public int Player(string history) => history.Length % 2;
    public int[] Actions() => new[]
    {
        0,
        1,
        2
    };
}

// Le prédicteur lit les regrets déjà enregistrés sans modifier l'historique.
Func<Dictionary<string, List<double[]>>, string, double[]> exercice3 = (history, key) => null;
var observations = new Dictionary<string, List<double[]>>
{
    ["I0"] = new()
    {
        new[]
        {
            99.0,
            99,
            99
        },
        new[]
        {
            -2.0,
            1,
            0
        },
        new[]
        {
            2.0,
            -3,
            1
        },
        new[]
        {
            4.0,
            0,
            -1
        },
        new[]
        {
            0.0,
            3,
            2
        },
        new[]
        {
            1.0,
            1,
            1
        }
    },
    ["I1"] = new()
    {
        new[]
        {
            8.0,
            -1,
            2
        }
    },
    ["I3"] = new()
    {
        new[]
        {
            -1.0,
            4,
            6
        },
        new[]
        {
            3.0,
            -2,
            2
        }
    }
};
Dictionary<string, List<double[]>> Copies() => observations.ToDictionary(k => k.Key, k => k.Value.Select(v => (double[])v.Clone()).ToList());
var keys3 = new[]
{
    "I0",
    "I1",
    "I2",
    "I3"
};
var inputs3 = keys3.Select(key => Copies()).ToArray();
// Les instantanés permettent de détecter toute mutation des entrées par la réponse.
var snapshots3 = inputs3.Select(hist => System.Text.Json.JsonSerializer.Serialize(hist)).ToArray();
var results3 = keys3.Select((key, index) => exercice3(inputs3[index], key)).ToArray();
// Une réponse correcte ne dispense pas de conserver les observations passées.
if (Enumerable.Range(0, inputs3.Length).Any(index => snapshots3[index] != System.Text.Json.JsonSerializer.Serialize(inputs3[index])))
    throw new Exception("Le prédicteur ne doit pas modifier l’historique");
if (results3.All(r => r == null))
    Console.WriteLine("Exercice 3 à compléter.");
else
{
    var expected = new[]
    {
        new[]
        {
            1.4,
            1.0,
            .8
        },
        new[]
        {
            8.0,
            0,
            2
        },
        new[]
        {
            0.0,
            0,
            0
        },
        new[]
        {
            1.5,
            2,
            4
        }
    };
    if (results3.Any(r => r == null || r.Length != 3) || Enumerable.Range(0, 4).Any(i => Enumerable.Range(0, 3).Any(a => !double.IsFinite(results3[i][a]) || Math.Abs(results3[i][a] - expected[i][a]) > 1e-9)))
        throw new Exception("Prédiction incorrecte ou partielle");
    Console.WriteLine("Exercice 3 : sondes réussies.");
}


Exercice 3 à compléter.


## À retenir

Ce notebook compagnon de GT-13 illustre la **frontière théorie/pratique** du CFR optimiste — à condition de mesurer au lieu de réciter :

| Solveur | Garantie de regret | Exploitabilité mesurée ici (§2) |
|---|---|---|
| CFR | $\mathcal{O}(\sqrt{T})$ | la plus basse du banc |
| CFR+ | $\mathcal{O}(\sqrt{T})$ sous hypothèses usuelles | légèrement au-dessus de CFR à ce budget |
| Variante DCFR pédagogique | borne non établie ici | **la plus haute du banc** |
| Variante optimiste pédagogique (paramètres par défaut) | borne publiée non établie pour ce code | intermédiaire |
| Variante réglée empiriquement | borne non établie ici | non mesuré ici — c'est l'exercice 1 |

**Leçon** : le meilleur ordre asymptotique publié pour une construction stable-prédictive ne se traduit pas automatiquement en supériorité empirique sur petit jeu, **et la réputation empirique du DCFR ne suit pas une implémentation qui ne fait pas ce que son nom annonce**. Les deux écarts se mesurent ; aucun ne se récite.

**Sources complémentaires** :
- Farina, Kroer, Brown, Sandholm, *Stable-Predictive Optimistic Counterfactual Regret Minimization*, PMLR 97 (2019)
- Brown, Sandholm, *Solving Imperfect-Information Games via Discounted Regret Minimization*, **AAAI** 2019 (DCFR référence)
- Farina, Kroer, Sandholm, *Regret Circuits: Composability of Regret Minimizers*, ICML 2019 (socle réutilisé du carnet GT-13)

Voir aussi : `GameTheory-13-ImperfectInfo-CFR-Python.ipynb` — le carnet principal, dont ce notebook reprend le socle.

### Trois mesures à retenir

1. **Le classement mesuré n'est pas le classement théorique.** Sur les six donnes, l'exploitabilité en mode `simultaneous` vaut **0,006505** pour CFR, **0,006784** pour CFR+, **0,018525** pour OFTRL et **0,071657** pour DCFR : le DCFR, présenté comme le champion empirique, arrive **dernier** du banc.
2. **L'alternation dégrade tout le monde, sans réordonner.** En mode `alternating` (une donne par itération, donc six fois moins de mises à jour), les mêmes quatre solveurs donnent **0,018506**, **0,019493**, **0,022055** et **0,073547** : les valeurs augmentent et l'ordre des solveurs reste le même.
3. **Un diagnostic local ne mesure pas l’équilibre global.** La démonstration sur la seule donne J/Q amène Qp de **0,5 / 0,5** à environ **0,000502 / 0,999498** en 500 itérations. Le classement d’exploitabilité vient d’un autre protocole :1000 itérations sur six donnes et toutes les informations. Ces nombres ne prouvent donc pas une vitesse relative de convergence de la stratégie et de sa performance globale.


## Vérifiez votre compréhension

1. Le DCFR est présenté comme le champion empirique des CFR à mémoire pondérée. Que donne l'instrument de mesure du carnet — l'exploitabilité par meilleure réponse — sur les six donnes ?
2. Le mode `alternating` change-t-il le classement des solveurs, ou seulement les valeurs ?
3. Quel instrument le carnet utilise-t-il pour mesurer un solveur, et pourquoi pas un autre ?
4. La stratégie presque pure sur Qp dans la démonstration prouve-t-elle une convergence globale plus rapide ?

<details><summary>Réponses</summary>

1. Il arrive **dernier** : **0,071657** en `simultaneous` contre **0,006505** pour CFR, soit un ordre de grandeur d'écart. La réputation empirique du DCFR ne suit pas cette implémentation — le carnet le dit sans complaisance.
2. Le classement reste CFR < CFR+ < OFTRL < DCFR, mais les variations diffèrent selon le solveur (0,006505 → 0,018506 ; 0,071657 → 0,073547 pour le DCFR), parce que l'alternation réduit de six fois le nombre de mises à jour par itération.
3. Il mesure l'**exploitabilité réelle** par meilleure réponse (`get_exploitability()`). Une distance à l'uniforme classerait mal les stratégies : elle décrit la forme de la stratégie, pas sa profitabilité face à un adversaire qui réagit — l'énoncé de l'exercice 1 l'écarte donc explicitement.
4. Non. La démonstration à 500 itérations sur une seule donne mesure Qp ; le banc à1000 itérations sur six donnes mesure une exploitabilité globale. Un seul ensemble peut devenir presque pur tout en laissant d’autres décisions exploitables. Il faut aligner le protocole et mesurer toute la stratégie pour comparer des vitesses.

</details>


## Pour aller plus loin

- **Carnet suivant** : [GameTheory-13e — Révision de croyances](GameTheory-13e-Revision-Croyances-Adversaire-Python.ipynb) — du regret au modèle d'adversaire : la mise à jour porte désormais sur une croyance.
- **Dans la même série** : [GameTheory-13 — CFR](GameTheory-13-ImperfectInfo-CFR-Python.ipynb) — le socle (regret circuits, Deep CFR) réutilisé ici ; [GameTheory-13f — Ataraxos, le patron](GameTheory-13f-Ataraxos-Patron-Python.ipynb) — la même famille de mise à jour au temps de test.
- **Lectures** : Farina, Kroer, Brown & Sandholm (2019), *Stable-Predictive Optimistic Counterfactual Regret Minimization*, PMLR 97 — l'OFTRL et sa prédiction contrefactuelle ; Brown & Sandholm (2019), *Solving Imperfect-Information Games via Discounted Regret Minimization*, AAAI — le DCFR ; Farina, Kroer & Sandholm (2019), *Regret Circuits: Composability of Regret Minimizers*, ICML 2019 — le socle compositionnel.
